# Determining Solidus and Liquidus Temperatures with MAGEMin

This notebook demonstrates how to evaluate the solidus and liquidus temperatures for a bulk rock composition using MAGEMin, interfaced through Julia and Python. The approach includes both direct computational searches and packaged utility routines.

## 1. Setup and Initialization

Import necessary Python and Julia modules, and initialize the MAGEMin engine for subsequent calculations.

In [ ]:
import numpy as np
import juliacall
from juliacall import Main as jl, convert as jlconvert
import phasetools
from phasetools import MAGEMin_C

## 2. Define Bulk Composition and Calculation Parameters

For demonstration, we use the KLB-1 peridotite (Jennings & Holland, 2015) composition and the 'ig' database. Pressure is fixed at 8 kbar.

In [ ]:
db = "ig"  # 'ig' for igneous, Holland et al. (2018)
data = MAGEMin_C.Initialize_MAGEMin(db, verbose=False)
test = 0  # KLB-1
data = MAGEMin_C.use_predefined_bulk_rock(data, test)
P = 8.0  # Pressure in kbar

## 3. Brute-Force Search for Solidus and Liquidus

We estimate the onset and completion of melting by incrementing temperature and tracking the liquid fraction. The solidus is where liquid first appears; the liquidus is where liquid fraction reaches 1.

In [ ]:
liq_frac_vals = []
temp = np.linspace(1000, 2000, 51)
for T in temp:
    out = MAGEMin_C.single_point_minimization(P, T, data)
    liq_frac = phasetools.phase_frac(phase="liq", MAGEMinOutput=out, sys_in='mol')
    liq_frac_vals.append(liq_frac)

In [ ]:
# Identify solidus (last T with zero melt) and liquidus (first T with total melt)
liquidus_T = temp[np.where(np.array(liq_frac_vals) == 1.0)[0][0]]
solidus_T  = temp[np.where(np.array(liq_frac_vals) == 0.0)[0][-1]]

In [ ]:
import matplotlib.pyplot as plt
plt.plot(temp, liq_frac_vals, c='k', ls=':')
plt.scatter(liquidus_T, 1, marker='x', c='red', label='Liquidus')
plt.scatter(solidus_T, 0, marker='x', c='green', label='Solidus')
plt.xlabel('Temperature (°C)')
plt.ylabel('Liquid Fraction')
plt.legend()
plt.show()

In [ ]:
print(f'Solidus = {solidus_T}°C, Liquidus = {liquidus_T}°C')

## 4. Refined Degree-Resolution Search

To precisely locate the solidus and liquidus, repeat the minimisation at degree intervals near these transition points.

In [ ]:
# Refined search for solidus
initial_T = solidus_T + 10  # Start above approximate solidus
solidus_T_fine = float(initial_T)
out = MAGEMin_C.single_point_minimization(P, solidus_T_fine, data)
liq_frac = phasetools.phase_frac(phase="liq", MAGEMinOutput=out, sys_in='mol')
while liq_frac > 0:
    solidus_T_fine -= 1.0
    out = MAGEMin_C.single_point_minimization(P, solidus_T_fine, data)
    liq_frac = phasetools.phase_frac(phase="liq", MAGEMinOutput=out, sys_in='mol')
print(f'Refined solidus = {solidus_T_fine}°C')

In [ ]:
# Refined search for liquidus
initial_T = liquidus_T - 10  # Start below approximate liquidus
liquidus_T_fine = float(initial_T)
out = MAGEMin_C.single_point_minimization(P, liquidus_T_fine, data)
liq_frac = phasetools.phase_frac(phase="liq", MAGEMinOutput=out, sys_in='mol')
while liq_frac < 1.0:
    liquidus_T_fine += 1.0
    out = MAGEMin_C.single_point_minimization(P, liquidus_T_fine, data)
    liq_frac = phasetools.phase_frac(phase="liq", MAGEMinOutput=out, sys_in='mol')
print(f'Refined liquidus = {liquidus_T_fine}°C')

## 5. Using the root_scalar function from scipy instead

To precisely locate the solidus and liquidus even further


Root scalar documentation is found [here](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root_scalar.html)

In [ ]:
from scipy.optimize import root_scalar

def melt_fraction(T):
    out = MAGEMin_C.single_point_minimization(P, T, data)
    return phasetools.phase_frac(phase="liq", MAGEMinOutput=out, sys_in='mol')

# Find solidus: temperature where melt_fraction(T) crosses 0+
T_low, T_high = 1000, 2000  # Bracket based on prior knowledge or coarse scan

def solidus_func(T):
    return melt_fraction(T) - 1e-5  # or >0 if you want the very first appearance

solidus_T_rs = root_scalar(solidus_func, bracket=[T_low, T_high], method='bisect')


print(f"solidus: {solidus_T_rs.root} °C")

In [ ]:
def liquidus_func(T):
    return melt_fraction(T) - 1.0 + 1e-5


liquidus_T_rs = root_scalar(liquidus_func, bracket=[T_low, T_high], method='bisect')
print(f"Liquidus: {liquidus_T_rs.root} °C")

## 6. Using Packaged Utility Functions

The above approaches are encapsulated in convenience functions in the `PhaseFunctions` class.

In [ ]:
from phasetools import PhaseFunctions
phaseFunctions = PhaseFunctions(db=db, dataset=636)
phaseFunctions.setup_bulk_composition(list(out.oxides), list(out.bulk), sys_in='mol')

In [ ]:
solidus_T_util = phaseFunctions.find_phase_in(P, [1000, 2000], phase='liq', tol=1e-5, verbose=False)
print(f"Utility solidus: {solidus_T_util} °C")
liquidus_T_util = phaseFunctions.find_phase_saturation(P, [1000, 2000], phase='liq', tol=1e-5, verbose=False)
print(f"Utility liquidus: {liquidus_T_util} °C") 

## 6. Summary

- The solidus and liquidus are determined by tracking the liquid fraction over a temperature interval at fixed pressure and composition.
- Both brute-force and refined degree-resolution searches are demonstrated.
- Utility functions provide a streamlined interface for routine calculations.

For further details see:
- Jennings, E. S. & Holland, T. J. B. 2015. A Simple Thermodynamic Model for Melting of Peridotite in the System NCFMASOCr. Journal of Petrology 56, 869–892